In [ ]:
!pip install ifcopenshell
!pip install ifcopenshell openpyxl

In [ ]:
from google.colab import files  # ✅ Import the files module

print("📁 Please upload your IFC file...")
uploaded = files.upload()  # Opens file picker in Colab

# Get the uploaded file name
ifc_path = list(uploaded.keys())[0]
print(f"✅ Loaded file: {ifc_path}")


📁 Please upload your IFC file...


In [ ]:
import ifcopenshell
# Load the IFC file
ifc_file = ifcopenshell.open(ifc_path)



In [ ]:
import ifcopenshell.geom
import pandas as pd
from google.colab import files

print(ifcopenshell.version)

0.8.4


In [ ]:
# --- Function to get all project units ---
def get_units(ifc_file):
    unit_assignments = ifc_file.by_type("IfcUnitAssignment")
    units = {}
    if unit_assignments:
        for ua in unit_assignments:
            for unit in ua.Units:
                # Named units (like LengthUnit, AreaUnit, VolumeUnit)
                if unit.is_a("IfcSIUnit"):
                    units[unit.UnitType] = f"{unit.Prefix or ''}{unit.Name}"
                # Conversion-based units (like feet, inches)
                elif unit.is_a("IfcConversionBasedUnit"):
                    units[unit.UnitType] = unit.Name
    return units

# --- Function to show units for specific entity types ---
def show_entity_units(ifc_file, entity_name):
    entities = ifc_file.by_type(entity_name)
    units = get_units(ifc_file)
    print(f"\nEntity type: {entity_name}")
    print(f"Found {len(entities)} instances")
    print("Units in use:")
    for k, v in units.items():
        print(f"  {k}: {v}")

# Check units for IfcFlowSegment and IfcFlowFitting
show_entity_units(ifc_file, "IfcFlowSegment")
show_entity_units(ifc_file, "IfcFlowFitting")



# Geometry settings
settings = ifcopenshell.geom.settings()
settings.set(settings.USE_WORLD_COORDS, True)

# -------------------------------
# PIPES AND CABLES
# -------------------------------
def get_geom_length_diameter_segment(entity):
    length = None
    diameter = None
    radius = None
    if hasattr(entity, "Representation") and entity.Representation:
        for rep in entity.Representation.Representations:
            if rep.is_a("IfcShapeRepresentation"):
                for item in rep.Items:
                    if item.is_a("IfcExtrudedAreaSolid"):
                        if hasattr(item, "Depth"):
                            length = item.Depth
                        if item.SweptArea.is_a("IfcCircleProfileDef"):
                            radius = item.SweptArea.Radius
                            diameter = radius * 2
                        elif item.SweptArea.is_a("IfcRectangleProfileDef"):
                            diameter = max(item.SweptArea.XDim, item.SweptArea.YDim)
                            # no radius for rectangle
    return length, diameter, radius

def extract_cables_pipes(ifc_file):
    results = []
    for entity in ifc_file.by_type("IfcFlowSegment"):
        length, diameter, radius = get_geom_length_diameter_segment(entity)
        obj_type = getattr(entity, "ObjectType", None)
        results.append({
            "Category": "Pipe/Cable",
            "ObjectType": obj_type,
            "Length": round(length, 2) if length is not None else None,
            "Diameter": round(diameter, 2) if diameter is not None else None,
            "Radius": round(radius, 2) if radius is not None else None
        })
    return results

# -------------------------------
# FITTINGS
# -------------------------------
def get_geom_length_diameter_fitting(entity):
    length = None
    diameter = None
    try:
        shape = ifcopenshell.geom.create_shape(settings, entity)
        verts = shape.geometry.verts
        xs = verts[0::3]; ys = verts[1::3]; zs = verts[2::3]
        dx = max(xs) - min(xs); dy = max(ys) - min(ys); dz = max(zs) - min(zs)
        length = max(dx, dy, dz)
        diameter = max(dx, dy)
    except Exception:
        pass
    return length, diameter

def extract_fittings(ifc_file):
    results = []
    for entity in ifc_file.by_type("IfcFlowFitting"):
        length, diameter = get_geom_length_diameter_fitting(entity)
        obj_type = getattr(entity, "ObjectType", None)
        results.append({
            "Category": "Fitting",
            "ObjectType": obj_type,
            "Length": round(length, 2) if length is not None else None,
            "Radius": None,   # fittings don’t use radius
            "Diameter": None  # fittings not split by diameter
        })
    return results

# -------------------------------
# RUN EXTRACTION
# -------------------------------

# Pipes/cables summary (group by ObjectType + Diameter + Radius)
data_segments = extract_cables_pipes(ifc_file)
df_segments = pd.DataFrame(data_segments)
summaryflow = df_segments.groupby(["Category","ObjectType","Diameter","Radius"], dropna=False)["Length"].sum().reset_index()

# Fittings summary (group only by ObjectType)
data_fittings = extract_fittings(ifc_file)
df_fittings = pd.DataFrame(data_fittings)
summaryfittings = df_fittings.groupby(["Category","ObjectType"], dropna=False)["Length"].sum().reset_index()

# Merge both summaries into one sheet
combined_summary = pd.concat([summaryflow, summaryfittings], ignore_index=True)

# Add units row at the top
units_row = pd.DataFrame([{
    "Category": "Units:",
    "ObjectType": "",
    "Diameter": "",
    "Radius": "",
    "Length": ""
}])
combined_summary = pd.concat([units_row, combined_summary], ignore_index=True)

print("Combined summary (pipes/cables + fittings, fittings not split by diameter):")
print(combined_summary)

# Export to Excel
combined_summary.to_excel("summary_output.xlsx", index=False, engine="openpyxl")
files.download("summary_output.xlsx")
